GRADIENT BOOSTING NO PCA WITH LASSO FEATURES

Jack Silkaitis

In [14]:
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor

from sklearn.metrics import mean_squared_error, mean_absolute_error
import plotly.express as px
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, cross_validate
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.decomposition import PCA

In [16]:
# Import data
# df = pd.read_csv('./LassoFeatures.csv')
df = pd.read_csv('./LassoCoeffs2.csv')
df.head(3)
df.shape

(22099, 2)

In [17]:
df.columns

Index(['Unnamed: 0', '0'], dtype='object')

In [5]:
# Split and prepare data
(df_train,df_test) = train_test_split(df,train_size=0.8,
                                      test_size=0.2,
                                      random_state=0)

In [6]:
X_train = df_train.drop(['log radius','Unnamed: 0'],axis=1)
y_train = df_train['log radius']
X_test = df_test.drop(['log radius','Unnamed: 0'],axis=1)
y_test = df_test['log radius']

In [7]:
stnd = StandardScaler().set_output(transform='pandas')

X_number = X_train.select_dtypes(include='number')
X_categorical = X_train.select_dtypes(exclude='number')
X_number = stnd.fit_transform(X_number)
X_train = pd.concat([X_number, X_categorical], axis=1)

X_number = X_test.select_dtypes(include='number')
X_categorical = X_test.select_dtypes(exclude='number')
X_number = stnd.transform(X_number)
X_test = pd.concat([X_number, X_categorical], axis=1)

In [9]:
tree = DecisionTreeRegressor(max_depth=2)
tree.fit(X_train,y_train)
tree.score(X_test, y_test)

0.15162466895652094

In [10]:
B = np.arange(1,301,100)
C = np.arange(1,10,3)
D = np.arange(0.01,1,0.33)
grid = {'n_estimators':B, 'max_depth':C, 'learning_rate':D}

gbt = GradientBoostingRegressor()
gbtCV = GridSearchCV(gbt,param_grid=grid,return_train_score=True,n_jobs=-1, verbose=2)
gbtCV.fit(X_train,y_train)

print('best params =',gbtCV.best_params_, '  valid acc =',gbtCV.best_score_)

Fitting 5 folds for each of 27 candidates, totalling 135 fits
best params = {'learning_rate': np.float64(0.34), 'max_depth': np.int64(7), 'n_estimators': np.int64(201)}   valid acc = 0.8438815548737553


In [11]:
B = np.arange(150,300,50)
C = np.arange(6,9,1)
D = np.arange(.2, .5, .1)
grid = {'n_estimators':B, 'max_depth':C, 'learning_rate':D}

gbt = GradientBoostingRegressor()
gbtCV = GridSearchCV(gbt,param_grid=grid,return_train_score=True,n_jobs=-1, verbose=2)
gbtCV.fit(X_train,y_train)

print('best params =',gbtCV.best_params_, '  valid acc =',gbtCV.best_score_)

Fitting 5 folds for each of 27 candidates, totalling 135 fits
best params = {'learning_rate': np.float64(0.2), 'max_depth': np.int64(7), 'n_estimators': np.int64(250)}   valid acc = 0.8567340455410731


In [15]:
B = np.arange(225,301,25)
C = np.arange(.1,.3,.05)
grid = {'n_estimators':B, 'learning_rate':C}

gbt = GradientBoostingRegressor(max_depth=7)
gbtCV = GridSearchCV(gbt,param_grid=grid,return_train_score=True,n_jobs=-1, verbose=2)
gbtCV.fit(X_train,y_train)

print('best params =',gbtCV.best_params_, '  valid acc =',gbtCV.best_score_)

Fitting 5 folds for each of 16 candidates, totalling 80 fits
best params = {'learning_rate': np.float64(0.20000000000000004), 'n_estimators': np.int64(275)}   valid acc = 0.8588672737822367


In [9]:
bestGBT = GradientBoostingRegressor(max_depth=7, n_estimators=275, learning_rate=.2)

In [23]:
print('Test R2',gbtCV.score(X_test,y_test))
print('Train R2',gbtCV.score(X_train,y_train))

Test R2 0.9233374702545136
Train R2 0.9899178760263251


In [12]:
bestGBT.fit(X_train, y_train)
y_pred = bestGBT.predict(X_test)
mean_squared_error(y_pred, y_test)

0.006616281418587937

In [24]:
y_pred = gbtCV.predict(X_train)
mean_squared_error(y_pred, y_train)

0.0008455521879191276

In [22]:
mean_absolute_error(y_pred,y_test)

0.041857384205628366

In [29]:
np.sqrt(mean_squared_error(10**y_pred, 10**y_test))

np.float64(29.715595742883515)

In [13]:
np.mean(np.abs(10**(y_pred) - 10**y_test) / 10**y_test)

np.float64(0.10271785930086158)